<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`:
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
# Google Colab includes Pandas and normally includes Requests and BeautifulSoup.
# Only uncomment this line if imports fail:
# %pip install -q beautifulsoup4 requests pandas


In [2]:
import re
import unicodedata
import requests
from bs4 import BeautifulSoup
import pandas as pd


and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def clean_cell_text(cell):
    """Read visible text, removing footnote markers without changing the source HTML."""
    copy = BeautifulSoup(str(cell), "html.parser")
    for sup in copy.find_all("sup"):
        sup.decompose()
    text = copy.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\(\s+", "(", text)
    return re.sub(r"\s+\)", ")", text)


def date_time(table_cell):
    """Obtain date and time from the two text fragments in the Wikipedia cell."""
    copy = BeautifulSoup(str(table_cell), "html.parser")
    for sup in copy.find_all("sup"):
        sup.decompose()
    parts = [text.strip() for text in copy.stripped_strings if text.strip()]
    return (parts + ["", ""])[:2]


def booster_version(table_cell):
    """Return the first-stage booster description even without a hyperlink."""
    return clean_cell_text(table_cell)


def landing_status(table_cell):
    """Return the booster landing result, excluding citation markers."""
    return clean_cell_text(table_cell)


def get_mass(table_cell):
    """Extract a mass with units when present; otherwise preserve an available label."""
    mass = unicodedata.normalize("NFKD", clean_cell_text(table_cell))
    found = re.search(r"[\d,.]+\s*kg", mass, flags=re.IGNORECASE)
    return found.group(0) if found else (mass or None)


def extract_column_from_header(header_cell):
    """Read a <th> label without removing the label's meaningful link text."""
    label = clean_cell_text(header_cell)
    return label if label and not label.isdigit() else None


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
# TASK 1: Request the 2021 historical Wikipedia revision, as required by IBM.
response = requests.get(static_url, headers=headers, timeout=40)
response.raise_for_status()
print("HTTP status:", response.status_code)
print("HTML characters received:", len(response.text))


HTTP status: 200
HTML characters received: 3289905


Create a `BeautifulSoup` object from the HTML `response`


In [6]:
# Parse the downloaded response with BeautifulSoup.
soup = BeautifulSoup(response.text, "html.parser")


Print the page title to verify if the `BeautifulSoup` object was created properly


In [7]:
# Display the HTML page title to verify the response.
print(soup.title)


<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
# TASK 2: This definition was missing in the original notebook.
# Collect ALL HTML tables; the next cell selects the relevant launch table.
html_tables = soup.find_all("table")
print("HTML tables found:", len(html_tables))
if not html_tables:
    raise RuntimeError("No HTML tables found. Check whether Wikipedia returned the expected historical article.")


HTML tables found: 29


Starting from the third table is our target table contains the actual launch records.


In [9]:
# Find the historical launch tables from their column headers. This is safer
# than depending on the third table's position, which can vary with the HTML.
def has_launch_header(table):
    for tr in table.find_all("tr"):
        labels = [clean_cell_text(th).lower() for th in tr.find_all("th", recursive=False)]
        if any("flight no" in label for label in labels) and any("payload" in label for label in labels):
            return True
    return False

launch_tables = [table for table in html_tables if has_launch_header(table)]
if not launch_tables:
    raise RuntimeError("Launch-record tables were not found in the downloaded HTML.")
first_launch_table = launch_tables[0]
print("Launch-record tables:", len(launch_tables))
print(first_launch_table.prettify()[:2500])  # Preview instead of flooding the notebook


Launch-record tables: 9
<table class="wikitable plainrowheaders collapsible" id="mwmQ" style="width: 100%;">
 <tbody id="mwmg">
  <tr id="mwmw">
   <th id="mwnA" scope="col">
    Flight No.
   </th>
   <th id="mwnQ" scope="col">
    Date and
    <br id="mwng"/>
    time (
    <a href="https://en.wikipedia.org/wiki/Coordinated_Universal_Time" id="mwnw" rel="mw:WikiLink" title="Coordinated Universal Time">
     UTC
    </a>
    )
   </th>
   <th id="mwoA" scope="col">
    <a href="https://en.wikipedia.org/wiki/List_of_Falcon_9_first-stage_boosters" id="mwoQ" rel="mw:WikiLink" title="List of Falcon 9 first-stage boosters">
     Version,
     <br id="mwog"/>
     Booster
    </a>
    <sup about="#mwt70" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":"booster","group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-booster-11"},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"name":{"wt":"booster"},"1":{"wt":"Falcon 9 first-stage boost

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
# Extract all named columns from the first launch table's header row.
column_names = []
for tr in first_launch_table.find_all("tr"):
    labels = [clean_cell_text(th).lower() for th in tr.find_all("th", recursive=False)]
    if any("flight no" in label for label in labels) and any("payload" in label for label in labels):
        for th in tr.find_all("th", recursive=False):
            name = extract_column_from_header(th)
            if name is not None and len(name) > 0:
                column_names.append(name)
        break

print("Columns found:", len(column_names))
assert column_names, "Header extraction returned no column names."


Columns found: 10


Check the extracted column names


In [11]:
print(column_names)

['Flight No.', 'Date and time (UTC)', 'Version, Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
# TASK 3: The Date and time field is split into Date and Time.
# Keep column names compatible with the expected IBM scraping exercise.
launch_dict = {
    "Flight No.": [],
    "Date": [],
    "Time": [],
    "Version Booster": [],
    "Launch site": [],
    "Payload": [],
    "Payload mass": [],
    "Orbit": [],
    "Customer": [],
    "Launch outcome": [],
    "Booster landing": [],
}
print("Output columns:", list(launch_dict))


Output columns: ['Flight No.', 'Date', 'Time', 'Version Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [13]:
# Parse each historic launch table. Numbered <th> cells represent launch rows.
# Extract nine <td> fields and append every value to launch_dict.
extracted_row = 0
for table in launch_tables:
    for tr in table.find_all("tr"):
        th = tr.find("th", recursive=False)
        if th is None:
            continue
        # Ignore year headings, subheadings, and citations.
        flight_number = next(iter(th.stripped_strings), "").strip()
        if not flight_number.isdigit():
            continue
        cells = tr.find_all("td", recursive=False)
        if len(cells) < 9:
            continue

        date, time = date_time(cells[0])
        record = {
            "Flight No.": flight_number,
            "Date": date.strip(", "),
            "Time": time,
            "Version Booster": booster_version(cells[1]),
            "Launch site": clean_cell_text(cells[2]),
            "Payload": clean_cell_text(cells[3]),
            "Payload mass": get_mass(cells[4]),
            "Orbit": clean_cell_text(cells[5]),
            "Customer": clean_cell_text(cells[6]),
            "Launch outcome": clean_cell_text(cells[7]),
            "Booster landing": landing_status(cells[8]),
        }
        for key, value in record.items():
            launch_dict[key].append(value)
        extracted_row += 1

assert extracted_row > 0, "No numbered launch records found. Review the historical HTML structure."
assert len({len(values) for values in launch_dict.values()}) == 1, "Inconsistent column lengths."
print("Launch records extracted:", extracted_row)


Launch records extracted: 121


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [14]:
# Convert the scraped data to a DataFrame and verify its first five rows.
df = pd.DataFrame(launch_dict)
print("DataFrame size:", df.shape)
print("Missing values per column:")
print(df.isna().sum())
display(df.head())


DataFrame size: (121, 11)
Missing values per column:
Flight No.         0
Date               0
Time               0
Version Booster    0
Launch site        0
Payload            0
Payload mass       2
Orbit              0
Customer           0
Launch outcome     0
Booster landing    0
dtype: int64


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,4 June 2010,18:45,F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,None,LEO,SpaceX,Success,Failure (parachute)
1,2,8 December 2010,15:43,F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),None,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,3,22 May 2012,07:44,F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),525 kg,LEO (ISS),NASA (COTS),Success,No attempt
3,4,8 October 2012,00:35,F9 v1.0 B0006.1,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg",LEO (ISS),NASA (CRS),Success,No attempt
4,5,1 March 2013,15:10,F9 v1.0 B0007.1,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg",LEO (ISS),NASA (CRS),Success,No attempt


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab.

Following labs will be using a provided dataset to make each lab independent.


In [15]:
# Export the complete scraped data as CSV.
df.to_csv("spacex_web_scraped.csv", index=False)
print("Saved: spacex_web_scraped.csv —", df.shape[0], "rows")

Saved: spacex_web_scraped.csv — 121 rows


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
